# Port Keypoint Detection — Training Notebook

Trains a CNN to predict 5 keypoints on the SFP or SC port face from a single RGB image.
Outputs are used with `cv2.solvePnP` to recover the port's 6-DoF pose and publish a TF frame
that feeds the existing `_gt_approach` alignment controller.

## Running on Google Colab

1. **Set runtime to GPU**: Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro)
2. **Prepare your data zip**: On your local machine, zip the `pose_data/` directory:
   ```bash
   cd ~/ws_aic/src/aic/my_policy_node
   zip -r pose_data.zip pose_data/
   ```
3. **Upload to Google Drive**: Put `pose_data.zip` anywhere in your Drive, e.g. `My Drive/aic/`
4. Run cells top-to-bottom — Colab cells handle Drive mount and extraction automatically
5. After training, model files are saved back to Drive automatically

## Running on a university cluster (JupyterHub)

Skip the two Colab-specific cells (they detect Colab and do nothing on a cluster).
Upload `pose_data/` alongside this notebook and run top-to-bottom.

| Plug | Strategy | Approx. training time (1× A100) |
|------|----------|---------------------------------|
| SFP  | Full 576×512 image | ~8 min (150 epochs) |
| SC   | 96×96 crop centred on port | ~4 min (150 epochs) |

In [ ]:
# ── Colab: GPU check + Drive mount ───────────────────────────────────────────
# (Safe to run on any platform — does nothing outside Colab)
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    r = subprocess.run(
        "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
        shell=True, capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"GPU: {r.stdout.strip()} ✓")
    else:
        print("⚠  No GPU detected.  Go to: Runtime → Change runtime type → GPU")

    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
else:
    print(f"Running outside Colab (sys.argv[0]={sys.argv[0]}) — Drive mount skipped")

In [ ]:
# ── Colab: Extract pose_data from Drive ──────────────────────────────────────
# Update DRIVE_ZIP_PATH to wherever you uploaded pose_data.zip in your Drive.
# This cell is a no-op on a cluster (IN_COLAB is False).

if IN_COLAB:
    from pathlib import Path
    import zipfile

    DRIVE_ZIP_PATH = "/content/drive/MyDrive/aic/pose_data.zip"   # ← UPDATE THIS

    _extract_to = Path("/content")
    if not (_extract_to / "pose_data").exists():
        print(f"Extracting {DRIVE_ZIP_PATH} …")
        with zipfile.ZipFile(DRIVE_ZIP_PATH) as zf:
            zf.extractall(_extract_to)
        print("Extraction complete.")
    else:
        print("pose_data/ already present at /content/pose_data ✓")

    _colab_data_root  = _extract_to / "pose_data"
    _colab_output_dir = _extract_to / "kp_model_output"
    _colab_output_dir.mkdir(exist_ok=True)
    print(f"Data  : {_colab_data_root}")
    print(f"Output: {_colab_output_dir}")

In [ ]:
# Install / verify packages
import subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args, "-q"])

try:
    import cv2
except ImportError:
    pip("opencv-python-headless")

try:
    from tqdm.auto import tqdm
except ImportError:
    pip("tqdm")

import torch, torchvision
print(f"torch {torch.__version__}  torchvision {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os, json, random, math
import numpy as np
import cv2
import matplotlib
# Use inline backend on Colab/Jupyter; Agg on headless cluster
try:
    get_ipython  # type: ignore
    # Running inside Jupyter — leave backend as-is (inline is already active)
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Configuration — edit these before running

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
if IN_COLAB:
    # Paths set automatically by the Colab extraction cell above
    DATA_ROOT  = _colab_data_root
    OUTPUT_DIR = _colab_output_dir
else:
    # UPDATE THESE FOR YOUR CLUSTER ENVIRONMENT
    DATA_ROOT  = Path("../pose_data")        # root of pose_data/
    OUTPUT_DIR = Path("../kp_model_output")  # checkpoints, plots, saved model

# ── Training target ──────────────────────────────────────────────────────────
PLUG_TYPE = "sfp"     # "sfp" or "sc" — train one at a time

# Image dimensions fed to the network
# SFP: use half-res full image.  SC: uses crop (set below), so these are crop output size.
IMG_H = 512;  IMG_W = 576   # for SFP
# IMG_H = 256;  IMG_W = 256 # for SC crop — uncomment when training SC

# SC crop parameters (ignored for SFP)
SC_CROP_CONTEXT_PX = 80   # context around port in ORIGINAL-resolution pixels (port spread ~4px)

N_KP = 5

# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE    = 16
EPOCHS        = 150
WARMUP_EPOCHS = 25    # head-only (frozen backbone) warmup
LR            = 1e-3
LR_FINETUNE   = 3e-5  # learning rate after backbone is unfrozen
LR_MIN        = 1e-6
WEIGHT_DECAY  = 1e-4
DROPOUT       = 0.40
VAL_FRAC      = 0.15
SEED          = 42

# DataLoader workers: 2 on Colab (fewer CPU cores), 4 on cluster
NUM_WORKERS = 2 if IN_COLAB else 4

# ── Augmentation ─────────────────────────────────────────────────────────────
AUG_ROTATE_DEG = 8.0
AUG_SCALE      = (0.85, 1.15)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Plug type   : {PLUG_TYPE}")
print(f"Image size  : {IMG_W}×{IMG_H}")
print(f"Data root   : {DATA_ROOT}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Num workers : {NUM_WORKERS}")

## Load data

In [ ]:
def load_dataset(data_root: Path, plug_type: str):
    label_dir = data_root / plug_type / "labels"
    samples = []
    for lf in sorted(label_dir.glob("*.json")):
        with open(lf) as f:
            lbl = json.load(f)
        img_path = data_root / plug_type / lbl["image"]
        if not img_path.exists():
            continue
        samples.append({
            "img_path" : str(img_path),
            "kp2d"     : np.array(lbl["keypoints_2d"], dtype=np.float32),
            "kp3d"     : np.array(lbl["keypoints_3d"], dtype=np.float32),
            "port_pos" : np.array(lbl["port_pos"],     dtype=np.float32),
            "port_R"   : np.array(lbl["port_R"],       dtype=np.float32),
            "orig_w"   : lbl["img_width"],
            "orig_h"   : lbl["img_height"],
        })
    print(f"Loaded {len(samples)} samples for '{plug_type}'")
    return samples

samples = load_dataset(DATA_ROOT, PLUG_TYPE)

## Visualise samples and check keypoint distribution

In [ ]:
def draw_keypoints_on(img_rgb, kp2d, radius=6):
    img = img_rgb.copy()
    colors = [(255,50,50),(50,255,50),(50,50,255),(255,255,50),(255,50,255)]
    for i, (u, v) in enumerate(kp2d):
        cv2.circle(img, (int(round(u)), int(round(v))), radius, colors[i], -1)
        cv2.putText(img, str(i), (int(u)+radius+2, int(v)+radius+2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[i], 1)
    return img

vis_samples = random.sample(samples, min(8, len(samples)))
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, s in zip(axes.flatten(), vis_samples):
    img = cv2.imread(s["img_path"])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    kp = s["kp2d"]
    cx, cy = kp[:,0].mean(), kp[:,1].mean()
    spread = max(float(kp[:,0].max()-kp[:,0].min()), float(kp[:,1].max()-kp[:,1].min()), 5)
    pad = spread * 12
    x1 = max(0, int(cx-pad)); y1 = max(0, int(cy-pad))
    x2 = min(img.shape[1], int(cx+pad)); y2 = min(img.shape[0], int(cy+pad))
    crop = draw_keypoints_on(img_rgb, kp)[y1:y2, x1:x2]
    ax.imshow(crop); ax.axis("off")
    ax.set_title(f"center=({cx:.0f},{cy:.0f})  spread={spread:.1f}px", fontsize=8)
plt.suptitle(f"{PLUG_TYPE.upper()} samples — cropped around port", fontsize=13)
plt.tight_layout()
out = OUTPUT_DIR / f"{PLUG_TYPE}_samples_preview.png"
plt.savefig(out, dpi=100); plt.show()
print(f"Saved → {out}")

In [ ]:
# Distribution statistics
kp_all  = np.array([s["kp2d"] for s in samples])   # (N,5,2)
cx_all  = kp_all[:,0,0]
cy_all  = kp_all[:,0,1]
spreads = np.array([float(max(s["kp2d"][:,0].max()-s["kp2d"][:,0].min(),
                               s["kp2d"][:,1].max()-s["kp2d"][:,1].min()))
                    for s in samples])
ow, oh = samples[0]["orig_w"], samples[0]["orig_h"]

print(f"=== {PLUG_TYPE.upper()} — original resolution {ow}×{oh} ===")
print(f"  center u : {cx_all.min():.0f} – {cx_all.max():.0f}  (mean {cx_all.mean():.0f} ± {cx_all.std():.0f})")
print(f"  center v : {cy_all.min():.0f} – {cy_all.max():.0f}  (mean {cy_all.mean():.0f} ± {cy_all.std():.0f})")
print(f"  kp spread: min={spreads.min():.1f}  max={spreads.max():.1f}  mean={spreads.mean():.1f}px")
if PLUG_TYPE == "sc":
    print()
    print("  SC keypoint spread is ~4px at full resolution.")
    print(f"  At crop output size ({IMG_W}×{IMG_H}), effective spread ≈ ",
          f"{spreads.mean() / SC_CROP_CONTEXT_PX * IMG_W:.0f}px — viable for regression.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(cx_all, cy_all, s=8, alpha=0.5); ax1.invert_yaxis()
ax1.set(xlabel="u", ylabel="v", title="Port center location in image")
ax2.hist(spreads, bins=30)
ax2.set(xlabel="KP spread (px)", ylabel="Count", title="Keypoint spread per sample")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{PLUG_TYPE}_kp_distribution.png", dpi=100)
plt.show()

## Dataset and augmentation

**SFP** — full image resized to 576×512.  Keypoint spread ~14px at that resolution.

**SC** — 96px context crop centred on the port → resized to 256×256.  This zooms in ~32×
and brings the 4px spread to ~17px, making it detectable.  During inference the crop centre
comes from the robot TCP position projected into the image.

In [ ]:
class WingLoss(nn.Module):
    """Wing loss — better gradient for small keypoint errors than MSE."""
    def __init__(self, w: float = 10.0, epsilon: float = 2.0):
        super().__init__()
        self.w = w; self.eps = epsilon
        self.C = w - w * math.log(1.0 + w / epsilon)

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        diff = (pred - target).abs()
        return torch.where(
            diff < self.w,
            self.w * torch.log(1.0 + diff / self.eps),
            diff - self.C,
        ).mean()

In [ ]:
class KeypointDataset(Dataset):
    """
    Returns:
        img_t : (3, IMG_H, IMG_W) float32, ImageNet-normalised
        kp_t  : (N_KP*2,) float32, keypoint coords normalised to [0,1]
                relative to the network input image (after crop/resize)
    """
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, samples, plug_type, img_h, img_w,
                 sc_crop_px=SC_CROP_CONTEXT_PX, augment=True):
        self.samples   = samples
        self.plug_type = plug_type
        self.img_h     = img_h
        self.img_w     = img_w
        self.sc_crop   = sc_crop_px
        self.augment   = augment
        self.normalize = T.Normalize(mean=self.MEAN, std=self.STD)
        if augment:
            self.color_jitter = T.ColorJitter(
                brightness=0.35, contrast=0.35, saturation=0.25, hue=0.08
            )

    @staticmethod
    def _pil_crop_pad(img_pil, cx, cy, half):
        """Square crop centred on (cx,cy) with given half-width, zero-padding at borders."""
        W, H = img_pil.size
        x1c = cx - half; y1c = cy - half
        x1 = max(0, int(x1c)); y1 = max(0, int(y1c))
        x2 = min(W, int(x1c + half*2)); y2 = min(H, int(y1c + half*2))
        crop = img_pil.crop((x1, y1, x2, y2))
        pl = x1 - int(x1c); pt = y1 - int(y1c)
        side = int(half * 2)
        out = Image.new("RGB", (side, side), (0, 0, 0))
        out.paste(crop, (pl, pt))
        return out, int(x1c), int(y1c), side

    def _augment(self, img_pil, kp_norm):
        """kp_norm: (N,2) in [0,1].  Returns augmented (img_pil, kp_norm)."""
        import PIL.ImageFilter as PIF
        if random.random() < 0.8:
            img_pil = self.color_jitter(img_pil)
        if random.random() < 0.4:
            img_pil = img_pil.filter(PIF.GaussianBlur(radius=random.uniform(0.5, 1.5)))

        if AUG_ROTATE_DEG > 0 and random.random() < 0.6:
            angle = random.uniform(-AUG_ROTATE_DEG, AUG_ROTATE_DEG)
            img_pil = TF.rotate(img_pil, angle, interpolation=TF.InterpolationMode.BILINEAR)
            a = math.radians(-angle)
            cos_a, sin_a = math.cos(a), math.sin(a)
            du = kp_norm[:,0] - 0.5; dv = kp_norm[:,1] - 0.5
            kp_new = kp_norm.copy()
            kp_new[:,0] = 0.5 + cos_a*du - sin_a*dv
            kp_new[:,1] = 0.5 + sin_a*du + cos_a*dv
            if ((kp_new >= 0) & (kp_new <= 1)).all():
                kp_norm = kp_new

        if random.random() < 0.5:
            scale = random.uniform(AUG_SCALE[0], AUG_SCALE[1])
            W, H = img_pil.size
            nW, nH = int(W*scale), int(H*scale)
            img_pil = TF.resize(img_pil, (nH, nW), interpolation=TF.InterpolationMode.BILINEAR)
            if scale > 1.0:
                left = (nW-W)//2; top = (nH-H)//2
                img_pil = TF.crop(img_pil, top, left, H, W)
            else:
                pl = (W-nW)//2; pt = (H-nH)//2
                canvas = Image.new("RGB", (W, H), (0, 0, 0))
                canvas.paste(img_pil, (pl, pt))
                img_pil = canvas
            kp_norm[:,0] = kp_norm[:,0]*scale - (scale-1)/2
            kp_norm[:,1] = kp_norm[:,1]*scale - (scale-1)/2
            kp_norm = kp_norm.clip(0.0, 1.0)

        return img_pil, kp_norm

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_pil = Image.open(s["img_path"]).convert("RGB")
        orig_w, orig_h = img_pil.size

        if self.plug_type == "sc":
            kp = s["kp2d"]
            cx, cy = float(kp[:,0].mean()), float(kp[:,1].mean())
            img_pil, crop_x0, crop_y0, crop_side = self._pil_crop_pad(
                img_pil, cx, cy, self.sc_crop / 2
            )
            kp_norm = kp.copy()
            kp_norm[:,0] = (kp[:,0] - crop_x0) / crop_side
            kp_norm[:,1] = (kp[:,1] - crop_y0) / crop_side
            kp_norm = kp_norm.clip(0.0, 1.0)
        else:
            kp_norm = s["kp2d"].copy()
            kp_norm[:,0] /= orig_w
            kp_norm[:,1] /= orig_h

        if self.augment:
            img_pil, kp_norm = self._augment(img_pil, kp_norm)

        img_pil = TF.resize(img_pil, (self.img_h, self.img_w),
                            interpolation=TF.InterpolationMode.BILINEAR)
        img_t = self.normalize(TF.to_tensor(img_pil))
        kp_t  = torch.from_numpy(kp_norm.flatten().astype(np.float32))
        return img_t, kp_t

In [ ]:
random.seed(SEED); torch.manual_seed(SEED)
idx = list(range(len(samples))); random.shuffle(idx)
n_val   = max(1, int(len(samples) * VAL_FRAC))
train_s = [samples[i] for i in idx[:-n_val]]
val_s   = [samples[i] for i in idx[-n_val:]]

train_ds = KeypointDataset(train_s, PLUG_TYPE, IMG_H, IMG_W, augment=True)
val_ds   = KeypointDataset(val_s,   PLUG_TYPE, IMG_H, IMG_W, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  |  Steps/epoch: {len(train_loader)}")
imgs_b, kps_b = next(iter(train_loader))
print(f"Batch — images: {imgs_b.shape}  keypoints: {kps_b.shape}  "
      f"kp range [{kps_b.min():.3f}, {kps_b.max():.3f}]")

## Model — EfficientNet-B0 backbone + MLP regression head

In [ ]:
class KeypointNet(nn.Module):
    """EfficientNet-B0 feature extractor + regression head → N_KP×2 keypoints in [0,1]."""

    def __init__(self, n_kp: int = 5, dropout: float = 0.4):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features = backbone.features
        self.pool     = backbone.avgpool
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 512), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),  nn.SiLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, n_kp * 2),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.head(self.pool(self.features(x)))

model = KeypointNet(n_kp=N_KP, dropout=DROPOUT).to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: total={total:,}  trainable={trainable:,}")

with torch.no_grad():
    out = model(torch.randn(2, 3, IMG_H, IMG_W).to(DEVICE))
print(f"Output shape: {out.shape}  (expect [2, {N_KP*2}])")

## Training

Two phases:
1. **Warmup** (`WARMUP_EPOCHS`): backbone frozen, only regression head trained.
2. **Fine-tune** (remaining epochs): full network with 30× lower LR.

In [ ]:
criterion = WingLoss(w=10.0, epsilon=2.0)

def freeze_backbone(freeze: bool):
    for p in model.features.parameters():
        p.requires_grad = not freeze

def make_optimizer(lr):
    return optim.AdamW([p for p in model.parameters() if p.requires_grad],
                       lr=lr, weight_decay=WEIGHT_DECAY)

freeze_backbone(True)
optimizer = make_optimizer(LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=WARMUP_EPOCHS, eta_min=LR_MIN)

train_losses, val_losses = [], []
best_val, best_epoch = float("inf"), 0

def run_epoch(loader, train: bool) -> float:
    model.train(train)
    total = 0.0
    with torch.set_grad_enabled(train):
        for imgs, kps in loader:
            imgs, kps = imgs.to(DEVICE), kps.to(DEVICE)
            pred = model(imgs)
            loss = criterion(pred, kps)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item() * len(imgs)
    return total / len(loader.dataset)

pbar = tqdm(range(1, EPOCHS + 1), desc="Training")
for epoch in pbar:

    if epoch == WARMUP_EPOCHS + 1:
        freeze_backbone(False)
        optimizer = make_optimizer(LR_FINETUNE)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=LR_MIN * 0.1
        )
        tqdm.write(f"\n→ Phase 2: backbone unfrozen (lr={LR_FINETUNE})")

    tl = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    scheduler.step()
    train_losses.append(tl); val_losses.append(vl)

    if vl < best_val:
        best_val, best_epoch = vl, epoch
        torch.save(model.state_dict(), OUTPUT_DIR / f"{PLUG_TYPE}_best.pth")

    pbar.set_postfix({"train": f"{tl:.4f}", "val": f"{vl:.4f}",
                      "best": f"{best_val:.4f}@{best_epoch}"})
    if epoch % 25 == 0:
        cur_lr = optimizer.param_groups[0]["lr"]
        tqdm.write(f"  ep{epoch:3d}  train={tl:.5f}  val={vl:.5f}  lr={cur_lr:.2e}")

print(f"\nBest val loss: {best_val:.5f} at epoch {best_epoch}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, EPOCHS + 1)
for ax, yscale in [(ax1, "linear"), (ax2, "log")]:
    ax.plot(ep, train_losses, label="train")
    ax.plot(ep, val_losses,   label="val")
    ax.axvline(WARMUP_EPOCHS, color="gray",  ls=":",  alpha=0.7, label="finetune start")
    ax.axvline(best_epoch,    color="red",   ls="--", alpha=0.7, label=f"best ep{best_epoch}")
    ax.set(yscale=yscale, xlabel="Epoch", ylabel="Wing Loss", title=f"Loss ({yscale})")
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle(f"{PLUG_TYPE.upper()} training curves", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{PLUG_TYPE}_training_curves.png", dpi=100)
plt.show()

## Evaluation — pixel error on validation set

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / f"{PLUG_TYPE}_best.pth", map_location=DEVICE))
model.eval()

all_pred, all_true = [], []
with torch.no_grad():
    for imgs, kps in val_loader:
        all_pred.append(model(imgs.to(DEVICE)).cpu().numpy())
        all_true.append(kps.numpy())

pred_norm = np.vstack(all_pred).reshape(-1, N_KP, 2)
true_norm = np.vstack(all_true).reshape(-1, N_KP, 2)
pred_px   = pred_norm * np.array([IMG_W, IMG_H])
true_px   = true_norm * np.array([IMG_W, IMG_H])
errors    = np.linalg.norm(pred_px - true_px, axis=-1)   # (N, 5)

print(f"Validation pixel error (at {IMG_W}×{IMG_H})")
print(f"{'KP':>4}  {'Mean':>7}  {'Median':>7}  {'Std':>6}  {'P90':>6}")
for i in range(N_KP):
    e = errors[:,i]
    print(f"  {i:2d}   {e.mean():7.2f}  {np.median(e):7.2f}  {e.std():6.2f}  {np.percentile(e,90):6.2f}")
e_all = errors.flatten()
print(f" ALL   {e_all.mean():7.2f}  {np.median(e_all):7.2f}  {e_all.std():6.2f}  {np.percentile(e_all,90):6.2f}")

if PLUG_TYPE == "sc":
    scale = SC_CROP_CONTEXT_PX / IMG_W
    label = "original-res (via crop scale)"
else:
    scale = val_s[0]["orig_w"] / IMG_W
    label = f"original {val_s[0]['orig_w']}×{val_s[0]['orig_h']}"
print(f"\nEstimated error in {label}: {e_all.mean()*scale:.2f}px (mean)")

In [ ]:
model.eval()
norm_t = T.Normalize(mean=KeypointDataset.MEAN, std=KeypointDataset.STD)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for ax, s in zip(axes.flatten(), val_s[:8]):
    img = cv2.imread(s["img_path"])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig_w, orig_h = img.shape[1], img.shape[0]

    if PLUG_TYPE == "sc":
        kp = s["kp2d"]
        cx, cy = float(kp[:,0].mean()), float(kp[:,1].mean())
        img_pil, cx0, cy0, cside = KeypointDataset._pil_crop_pad(
            Image.fromarray(img_rgb), cx, cy, SC_CROP_CONTEXT_PX/2
        )
        true_px_s = np.array([(kp[:,0]-cx0)/cside, (kp[:,1]-cy0)/cside]).T * np.array([IMG_W, IMG_H])
    else:
        img_pil = Image.fromarray(img_rgb)
        true_px_s = (s["kp2d"] / np.array([orig_w, orig_h])) * np.array([IMG_W, IMG_H])

    img_in = TF.resize(img_pil, (IMG_H, IMG_W), interpolation=TF.InterpolationMode.BILINEAR)
    img_t  = norm_t(TF.to_tensor(img_in)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred_n = model(img_t).cpu().numpy().reshape(N_KP, 2)
    pred_px_s = pred_n * np.array([IMG_W, IMG_H])

    canvas = np.array(img_in)
    for u, v in true_px_s:
        cv2.circle(canvas, (int(u), int(v)), 4, (0, 220, 0), -1)
    for u, v in pred_px_s:
        cv2.circle(canvas, (int(u), int(v)), 4, (255, 100, 0), -1)

    cx_c  = (true_px_s[:,0].mean() + pred_px_s[:,0].mean()) / 2
    cy_c  = (true_px_s[:,1].mean() + pred_px_s[:,1].mean()) / 2
    spread = max(float(true_px_s[:,0].max()-true_px_s[:,0].min()),
                 float(true_px_s[:,1].max()-true_px_s[:,1].min()), 10)
    pad = spread * 10
    x1 = max(0, int(cx_c-pad)); y1 = max(0, int(cy_c-pad))
    x2 = min(IMG_W, int(cx_c+pad)); y2 = min(IMG_H, int(cy_c+pad))
    ax.imshow(canvas[y1:y2, x1:x2]); ax.axis("off")
    ax.set_title(f"err={np.linalg.norm(pred_px_s-true_px_s,axis=-1).mean():.1f}px", fontsize=8)

legend = [mpatches.Patch(color="green",  label="GT"),
          mpatches.Patch(color="orange", label="Pred")]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=11)
plt.suptitle(f"{PLUG_TYPE.upper()} — val predictions (green=GT, orange=pred)", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{PLUG_TYPE}_val_predictions.png", dpi=100)
plt.show()

## PnP pose evaluation

In [ ]:
KP3D = {
    "sfp": np.array([
        [ 0.000,  0.000, 0.0],   # 0 centre
        [-0.006, -0.004, 0.0],   # 1 top-left
        [ 0.006, -0.004, 0.0],   # 2 top-right
        [ 0.006,  0.004, 0.0],   # 3 bottom-right
        [-0.006,  0.004, 0.0],   # 4 bottom-left
    ], dtype=np.float64),
    "sc": np.array([
        [ 0.000,  0.000, 0.0],
        [ 0.0013, 0.000, 0.0],
        [ 0.000,  0.0013,0.0],
        [-0.0013, 0.000, 0.0],
        [ 0.000, -0.0013,0.0],
    ], dtype=np.float64),
}
kp3d_local = KP3D[PLUG_TYPE]

with open(DATA_ROOT / "camera_info.json") as f:
    cam = json.load(f)
K_orig = np.array(cam["K"], dtype=np.float64).reshape(3, 3)
K_net  = K_orig.copy()
ow, oh = cam["width"], cam["height"]

if PLUG_TYPE == "sc":
    su = IMG_W / SC_CROP_CONTEXT_PX; sv = IMG_H / SC_CROP_CONTEXT_PX
    K_net[0,0] *= su; K_net[0,2] = 0.5 * IMG_W
    K_net[1,1] *= sv; K_net[1,2] = 0.5 * IMG_H
else:
    K_net[0,0] *= IMG_W/ow; K_net[0,2] *= IMG_W/ow
    K_net[1,1] *= IMG_H/oh; K_net[1,2] *= IMG_H/oh

D = np.zeros(5, dtype=np.float64)
print(f"K_net (for {IMG_W}×{IMG_H} {PLUG_TYPE} input):\n{K_net}")

In [ ]:
t_errors, r_errors, reproj_errors = [], [], []

for s, p_norm, t_norm in zip(val_s, pred_norm, true_norm):
    p2d = (p_norm * np.array([IMG_W, IMG_H])).astype(np.float64)
    t2d = (t_norm * np.array([IMG_W, IMG_H])).astype(np.float64)
    ok_p, rv_p, tv_p = cv2.solvePnP(kp3d_local, p2d, K_net, D, flags=cv2.SOLVEPNP_ITERATIVE)
    ok_t, rv_t, tv_t = cv2.solvePnP(kp3d_local, t2d, K_net, D, flags=cv2.SOLVEPNP_ITERATIVE)
    if not (ok_p and ok_t): continue
    t_errors.append(float(np.linalg.norm(tv_p - tv_t)) * 1000.0)
    R_p, _ = cv2.Rodrigues(rv_p); R_t, _ = cv2.Rodrigues(rv_t)
    trace  = np.clip((np.trace(R_p @ R_t.T) - 1.0) / 2.0, -1.0, 1.0)
    r_errors.append(math.degrees(math.acos(trace)))
    proj, _ = cv2.projectPoints(kp3d_local, rv_p, tv_p, K_net, D)
    reproj_errors.append(float(np.linalg.norm(proj.squeeze() - p2d, axis=-1).mean()))

print(f"PnP evaluation — {len(t_errors)} val samples")
print(f"  Translation error : mean={np.mean(t_errors):.1f}mm  "
      f"median={np.median(t_errors):.1f}mm  p90={np.percentile(t_errors,90):.1f}mm")
print(f"  Rotation error    : mean={np.mean(r_errors):.2f}°  "
      f"median={np.median(r_errors):.2f}°  p90={np.percentile(r_errors,90):.2f}°")
print(f"  Reprojection error: mean={np.mean(reproj_errors):.2f}px at {IMG_W}×{IMG_H}")
print()
print("Viable for _gt_approach when:  translation < 5 mm  AND  rotation < 10°")

## Save model for deployment

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / f"{PLUG_TYPE}_best.pth", map_location=DEVICE))
model.eval()

with torch.no_grad():
    traced = torch.jit.trace(model, torch.randn(1, 3, IMG_H, IMG_W).to(DEVICE))
traced.save(str(OUTPUT_DIR / f"{PLUG_TYPE}_model_traced.pt"))

import shutil
shutil.copy(DATA_ROOT / "camera_info.json", OUTPUT_DIR / "camera_info.json")

cfg = {
    "plug_type"         : PLUG_TYPE,
    "img_h"             : IMG_H,
    "img_w"             : IMG_W,
    "n_keypoints"       : N_KP,
    "sc_crop_context_px": SC_CROP_CONTEXT_PX,
    "imagenet_mean"     : KeypointDataset.MEAN,
    "imagenet_std"      : KeypointDataset.STD,
    "orig_w"            : int(ow),
    "orig_h"            : int(oh),
    "best_epoch"        : int(best_epoch),
    "best_val_loss"     : float(best_val),
    "val_kp_error_px"   : float(e_all.mean()),
    "val_t_error_mm"    : float(np.mean(t_errors)),
    "val_r_error_deg"   : float(np.mean(r_errors)),
}
with open(OUTPUT_DIR / f"{PLUG_TYPE}_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print(f"Saved to {OUTPUT_DIR}/")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:45s}  {p.stat().st_size/1024:8.1f} KB")

In [ ]:
# ── Colab: copy model files back to Google Drive ──────────────────────────────
# Files are at /content/kp_model_output/ — copy to Drive so they survive session end.

if IN_COLAB:
    DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/aic/kp_model_output")   # ← UPDATE IF NEEDED
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

    for ext in ("*.pt", "*.pth", "*.json", "*.png"):
        for src in OUTPUT_DIR.glob(ext):
            shutil.copy(src, DRIVE_SAVE_DIR / src.name)

    print(f"Copied model files to Drive: {DRIVE_SAVE_DIR}")
    print("Files:")
    for p in sorted(DRIVE_SAVE_DIR.iterdir()):
        print(f"  {p.name:45s}  {p.stat().st_size/1024:8.1f} KB")
else:
    print("Not on Colab — skipping Drive copy.")
    print(f"Download the model files from: {OUTPUT_DIR}")

## After training — what to download and where to put it

**On Colab**: files are already saved to `My Drive/aic/kp_model_output/` by the cell above.

**On cluster**: download from `kp_model_output/`.

Place the following under `~/ws_aic/src/aic/my_policy_node/pose_model/`:

```
pose_model/
  sfp_model_traced.pt     ← TorchScript model (load with torch.jit.load)
  sfp_config.json         ← image size, normalisation, crop params
  sc_model_traced.pt      ← after running with PLUG_TYPE="sc"
  sc_config.json
  camera_info.json
```

The `PoseEstimator.py` policy node (next step) loads these files, runs keypoint detection,
solves PnP, and publishes the port pose as a TF frame so that `_gt_approach` works
unchanged with `ground_truth:=false`.

### SC at inference time

The SC model uses a 96px crop centred on the port.  At inference time the crop centre is
computed by projecting the robot TCP position + known offset into the camera image
(Option B, no extra detector needed).  This is implemented in `PoseEstimator.py`.